In [ ]:
! rm -r tb_logs/hp_search/

In [ ]:
from classifier import BertweetClassifier
from tweet_data_module import TweetsDataModule

import torch
import torch.nn as nn
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
import itertools
import pandas as pd

In [ ]:
SUMMARY_FILE = "search_summary.csv"
PARAM_SEARCH_LOG_NAME = "hp_search"
DEFAULT_TRIALS_PER_PARAMS = 5
EPOCHS = 75

datamodule_params = {
     "batch_size": 64, 
     "target_col": 'AR',
     "test_size":0.2, 
     "validation_size":0.2, 
     "oversample":True, 
     "random_state":2025
}

base_datamodule_params = {
     "target_col": 'AR',
     "test_size":0.2, 
     "validation_size":0.2,
     "random_state":2025
}

OUTPUT_COLUMNS = ["model_params", "data_params", "test_acc_AR", "test_acc_MB", "test_loss"]

In [ ]:
class ClassifierConstructor:
    def __init__(self, hidden_dim, dropout_p, activation):
        self.hidden_dim = hidden_dim
        self.dropout_p = dropout_p
        self.activation = activation
    
    def __call__(self, embedding_dim, num_labels):
        return nn.Sequential(
            nn.Linear(embedding_dim, self.hidden_dim),
            self.activation,
            nn.BatchNorm1d(self.hidden_dim),
            nn.Dropout(self.dropout_p),
            
            nn.Linear(self.hidden_dim, num_labels)
        )
    
    def __repr__(self):
        activ_name = ""
        if type(self.activation) is nn.ReLU:
            activ_name = "ReLU"
        elif type(self.activation) is nn.LeakyReLU:
            activ_name = "LeakyReLU"
        
        return str(
            {
                "hidden_dim": self.hidden_dim,
                "activation": activ_name,
                "dropout_p": self.dropout_p
            }
        )

In [ ]:
def evaluate_with_params(dataModule, model_params):
    model = BertweetClassifier(**model_params)
    model.id2label = dataModule.id2label          # list   -> e.g. ["neg","neu","pos"]
    model.label2id = dataModule.label2id
    logger = TensorBoardLogger("tb_logs", name=PARAM_SEARCH_LOG_NAME)
    trainer = pl.Trainer(max_epochs=EPOCHS, logger=logger, enable_checkpointing=False)
    trainer.fit(model=model, datamodule=dataModule)
    
    return trainer.test(model=model, datamodule=dataModule)      # <- returns a list of dicts

def param_search(param_iterator, trials_per_param = DEFAULT_TRIALS_PER_PARAMS, summary_file = "hp_search_summary.csv"):
    results_df = pd.DataFrame([], columns=OUTPUT_COLUMNS)

    for (model_params, data_params) in param_iterator:
        dataModule = TweetsDataModule.read_csv(**data_params)
        dataModule.setup("fit")

        if model_params["class_weights"] is not None:
            model_params["class_weights"] = dataModule.train_class_weights()

        for _ in range(trials_per_param):
            trial = evaluate_with_params(dataModule, model_params)
            results = trial[0]
            trial_results = {
                "model_params": str(model_params),
                "data_params": str(data_params),
                **results
            }
            results_df.loc[len(results_df)] = trial_results
        
        results_df.to_csv(summary_file)

# Will return a function (embedding_dim: int, num_labels: int) -> nn.Module
def one_layer_classifier_constructor(hidden_dim, dropout_p, activation_func):
    return ClassifierConstructor(hidden_dim, dropout_p, activation_func)

# Will return a function (embedding_dim: int, num_labels: int) -> nn.Module
def two_layer_classifier_constructor(hidden_dim1, hidden_dim2, dropout_p, activation_func):
    def constructor(embedding_dim, num_labels):
        return nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim1),
            activation_func,
            nn.BatchNorm1d(hidden_dim1),
            nn.Dropout(dropout_p),

            nn.Linear(hidden_dim1, hidden_dim2),
            activation_func,
            nn.BatchNorm1d(hidden_dim2),
            nn.Dropout(dropout_p),

            nn.Linear(hidden_dim2, num_labels)
        )

    return constructor

def create_model_options(model_name, learning_rate, freeze_encoder, classifier_constructor, class_weights):
    return {
        "transformer_model_name": model_name,
        "learning_rate": learning_rate,
        "freeze_encoder": freeze_encoder,
        "classifier_constructor": classifier_constructor,
        "class_weights": class_weights,
        "use_soft_labels":True,
    }

def create_data_options(batch_size, oversample):
    return {
        "batch_size": batch_size,
        "oversample": oversample,
        **base_datamodule_params,
    }

In [ ]:
MODEL_OPTIONS = [
    # "digio/Twitter4SSE",
    "peulsilva/sentence-transformer-trained-tweet",
    # "vinai/bertweet-base",
]
HIDDEN_DIM_OPTIONS = [64, 128, 256]
DROPOUT_OPTIONS = [0.3, 0.5]
ACTIVATION_FUNCTION_OPTIONS = [
    nn.ReLU(), nn.LeakyReLU(),
]
# ACTIVATION_FUNCTION_OPTIONS = [
#     nn.ReLU(), nn.Tanh(), nn.Hardswish(),
#     nn.LeakyReLU(),
# ]
LEARNING_RATE_OPTIONS = [1e-6, 1e-5, 1e-4, 1e-3]
FREEZE_ENCODER_OPTIONS = [True]
BATCH_SIZE_OPTIONS = [64]
OVERSAMPLE_OPTIONS = [False]
CLASS_WEIGHT_OPTIONS = [True]

def create_linear_param_iterator():
    # Do all linear layer options
    for (model, batch, oversample, freeze, lr, weight) in itertools.product(
        MODEL_OPTIONS, BATCH_SIZE_OPTIONS, OVERSAMPLE_OPTIONS, FREEZE_ENCODER_OPTIONS, LEARNING_RATE_OPTIONS, CLASS_WEIGHT_OPTIONS,
    ):
        # Skip over oversampling and class weights being done at the same time.
        if oversample and weight is not None:
            continue

        yield create_model_options(model, lr, freeze, None, weight), create_data_options(batch, oversample)

def create_one_hidden_layer_param_iterator():
    # Do all 1-hidden layer options
    for (model, hidden, dropout, activ, batch, oversample, freeze, lr, weight) in itertools.product(
        MODEL_OPTIONS, HIDDEN_DIM_OPTIONS, DROPOUT_OPTIONS, ACTIVATION_FUNCTION_OPTIONS,
        BATCH_SIZE_OPTIONS, OVERSAMPLE_OPTIONS, FREEZE_ENCODER_OPTIONS, LEARNING_RATE_OPTIONS, CLASS_WEIGHT_OPTIONS,
    ):
        if oversample and weight is not None:
            continue

        classifier = one_layer_classifier_constructor(hidden, dropout, activ)

        yield create_model_options(model, lr, freeze, classifier, weight), create_data_options(batch, oversample)

def create_two_hidden_layer_param_iterator():
    # Do all 2-hidden layer options
    for (model, hidden1, hidden2, dropout, activ, batch, oversample, freeze, lr, weight) in itertools.product(
        MODEL_OPTIONS, HIDDEN_DIM_OPTIONS, HIDDEN_DIM_OPTIONS, DROPOUT_OPTIONS, ACTIVATION_FUNCTION_OPTIONS,
        BATCH_SIZE_OPTIONS, OVERSAMPLE_OPTIONS, FREEZE_ENCODER_OPTIONS, LEARNING_RATE_OPTIONS, CLASS_WEIGHT_OPTIONS,
    ):
        if oversample and weight is not None:
            continue

        classifier = two_layer_classifier_constructor(hidden1, hidden2, dropout, activ)

        yield create_model_options(model, lr, freeze, classifier, weight), create_data_options(batch, oversample)

In [ ]:
torch.set_float32_matmul_precision('high')
# param_search(create_linear_param_iterator(), summary_file="hp_search_linear.csv")
param_search(create_one_hidden_layer_param_iterator(), summary_file="hp_search_1hidden.csv")
# param_search(create_two_hidden_layer_param_iterator(), summary_file="hp_search_2hidden.csv")